# Calibration 01 — Single Season Fit (2022-2023)

Aggregated (1-admdong) SEIRV 모델로 β_h, β_w, β_s, β_o, φ_a (14), γ_report (총 19 파라미터) fit.

1. 2022-2023 ILI target 시각화
2. Nelder-Mead optimizer 적용
3. L-BFGS-B optimizer 비교
4. Fit 결과 시각화 (관측 vs 예측)
5. Best fit 으로 2018-2019 시즌 예측 (검증)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from kt_epimodel.calibration.ili_target import (
    load_ili_target, simulation_to_ili,
)
from kt_epimodel.calibration.optimizer import (
    optimize_calibration, save_result, load_result,
)
from kt_epimodel.calibration.simple_model import (
    build_aggregated_inputs, simulate_aggregated,
)
from kt_epimodel.model.parameters import ModelParameters

OUT = Path('../outputs/calibration'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. ILI target (2022-2023)

In [ ]:
target = load_ili_target('2022-2023')
print(f"n_weeks: {target['n_weeks']}, valid: {target['is_valid'].sum()}")
print(f"range:   {target['ili_rate'].min():.2f} ~ {target['ili_rate'].max():.2f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(target['week_in_season'], target['ili_rate'], 'o-', label='ILI observed')
ax.set_xlabel('Week in season (0 = ISO 36)')
ax.set_ylabel('ILI rate (per 1000)')
ax.set_title('2022-2023 ILI target')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Nelder-Mead optimizer (max_iter=500)

In [ ]:
result_nm = optimize_calibration(
    season='2022-2023', method='Nelder-Mead',
    max_iterations=500, verbose=True,
)
print(f"\nNLL: {result_nm.nll_initial:.2f} -> {result_nm.nll:.2f}")
print(f"beta: h={result_nm.calibration.beta_h:.4f} w={result_nm.calibration.beta_w:.4f} s={result_nm.calibration.beta_s:.4f} o={result_nm.calibration.beta_o:.4f}")
print(f"gamma_report: {result_nm.calibration.gamma_report:.4f}")
save_result(result_nm, OUT / '2022-2023_NM.json')

## 3. L-BFGS-B optimizer

In [ ]:
result_lb = optimize_calibration(
    season='2022-2023', method='L-BFGS-B',
    max_iterations=200, verbose=True,
)
print(f"\nNLL: {result_lb.nll_initial:.2f} -> {result_lb.nll:.2f}")
save_result(result_lb, OUT / '2022-2023_LBFGSB.json')

## 4. Fit 결과 비교 (관측 vs 예측)

In [ ]:
inputs = build_aggregated_inputs()
pop_total = float(inputs['pop_15'].sum())

def predict_ili(result, t_span=(0, 224)):
    params = ModelParameters().with_calibration(result.calibration)
    sim = simulate_aggregated(
        params, inputs, seed_total=result.seed_total,
        initial_immunity=result.initial_immunity,
        initial_vaccinated_fraction=result.initial_vaccinated_fraction,
        t_span=t_span,
    )
    return simulation_to_ili(
        sim.daily_new_infection(), pop_total, result.calibration.gamma_report,
        n_weeks=target['n_weeks'],
    )

pred_nm = predict_ili(result_nm)
pred_lb = predict_ili(result_lb)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(target['week_in_season'], target['ili_rate'], 'o-', color='black', label='observed', linewidth=1.5)
ax.plot(target['week_in_season'], pred_nm, '-', color='C0', label=f'Nelder-Mead (NLL={result_nm.nll:.1f})', linewidth=2)
ax.plot(target['week_in_season'], pred_lb, '--', color='C1', label=f'L-BFGS-B (NLL={result_lb.nll:.1f})', linewidth=2)
ax.set_xlabel('Week in season'); ax.set_ylabel('ILI rate (per 1000)')
ax.set_title('2022-2023 fit: observed vs predicted')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'fit_2022-2023.png', dpi=120)
plt.show()

In [ ]:
# 파라미터 비교
import polars as pl
AGES = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70+']

rows = []
for r, name in [(result_nm, 'NM'), (result_lb, 'LBFGSB')]:
    rows.append({
        'method': name,
        'NLL': r.nll,
        'beta_h': r.calibration.beta_h,
        'beta_w': r.calibration.beta_w,
        'beta_s': r.calibration.beta_s,
        'beta_o': r.calibration.beta_o,
        'gamma_report': r.calibration.gamma_report,
        'n_evals': r.n_evaluations,
        'elapsed_s': r.elapsed_seconds,
    })
print(pl.DataFrame(rows))

# phi 비교
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(AGES, result_nm.calibration.phi, 'o-', label='NM')
ax.plot(AGES, result_lb.calibration.phi, 's--', label='LBFGSB')
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='ref (phi_25-29=1)')
ax.set_xlabel('Age group'); ax.set_ylabel('phi_a')
ax.set_title('Age-specific susceptibility (fitted)')
ax.tick_params(axis='x', rotation=45)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'phi_2022-2023.png', dpi=120)
plt.show()

## 5. Holdout 검증 — 2018-2019 시즌 예측

2022-2023 fit 결과를 그대로 2018-2019 에 적용해 예측. 시즌 간 일반화 확인.

In [ ]:
target_18 = load_ili_target('2018-2019')
params_best = ModelParameters().with_calibration(result_nm.calibration)
sim_18 = simulate_aggregated(
    params_best, inputs, seed_total=result_nm.seed_total,
    initial_immunity=result_nm.initial_immunity,
    initial_vaccinated_fraction=result_nm.initial_vaccinated_fraction,
    t_span=(0, 224),
)
pred_18 = simulation_to_ili(
    sim_18.daily_new_infection(), pop_total,
    result_nm.calibration.gamma_report, n_weeks=target_18['n_weeks'],
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(target_18['week_in_season'], target_18['ili_rate'], 'o-', color='black', label='2018-2019 observed', linewidth=1.5)
ax.plot(target_18['week_in_season'], pred_18, '-', color='C2', label='predicted (2022-2023 fit applied)', linewidth=2)
ax.set_xlabel('Week in season'); ax.set_ylabel('ILI rate (per 1000)')
ax.set_title('Holdout: 2022-2023 calibration -> 2018-2019 prediction')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'holdout_2018.png', dpi=120)
plt.show()

## 결론 + 다음 단계

**확인**:
- 19 파라미터 fit 동작 (Nelder-Mead, L-BFGS-B)
- 두 방법 결과 비교 — NLL 개선폭
- holdout 시즌 (2018-2019) 예측으로 일반화 정도 확인

**다음 단계** (Step b-2):
- 1D sensitivity sweep (각 β 별 NLL 곡선)
- 2D heatmap (β_w × β_o)
- 다중 시즌 동시 fit (Stage 3-2-c)